In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import os

DAYS_CSV = "/content/drive/MyDrive/aqi_project/data/raw/openaq_sensor_days_raw.csv"
LOC_CSV = "/content/drive/MyDrive/aqi_project/data/raw/openaq_locations.csv"

print("Daily data exists:", os.path.exists(DAYS_CSV))
print("Locations data exists:", os.path.exists(LOC_CSV))

Daily data exists: True
Locations data exists: True


In [ ]:
import pandas as pd

df = pd.read_csv(DAYS_CSV)

print(df.shape)
display(df.head())

(97220, 14)


,location_id,location_name,sensor_id,parameter,units,date,value_avg,min,max,sd,expected_count,observed_count,percent_complete,retrieved_at
0,17,"R K Puram, Delhi - DPCC",399,pm10,µg/m³,2018-01-01,491.0,216.000000,707.0,152.391785,24,13,54.0,2026-10-02T13:21:07+00:00
1,17,"R K Puram, Delhi - DPCC",399,pm10,µg/m³,2018-01-02,467.0,422.000000,521.0,46.788887,24,8,33.0,2026-10-02T13:21:07+00:00
2,17,"R K Puram, Delhi - DPCC",399,pm10,µg/m³,2018-01-03,427.0,357.000000,500.0,50.317741,24,11,46.0,2026-10-02T13:21:07+00:00
3,17,"R K Puram, Delhi - DPCC",399,pm10,µg/m³,2018-01-04,351.0,165.666667,554.0,121.128044,24,25,104.0,2026-10-02T13:21:07+00:00
4,17,"R K Puram, Delhi - DPCC",399,pm10,µg/m³,2018-01-05,407.0,285.000000,554.0,81.678627,24,24,100.0,2026-10-02T13:21:07+00:00


In [ ]:
import pandas as pd
df = pd.read_csv(DAYS_CSV, parse_dates=["date"])
print(df.shape, df["date"].min(), df["date"].max())
print(df["units"].value_counts(), df["parameter"].value_counts(), sep="\n")
print(df.duplicated(["sensor_id", "date"]).sum(), "duplicate sensor-days")
print((df["value_avg"] < 0).sum(), "negative values")
print(df.groupby(df["date"].dt.year)["location_id"].nunique())   # stations reporting per year
print(df.groupby(["location_id", "location_name", "parameter"])["date"].agg(["min", "max", "count"]).to_string())

(97220, 14) 2018-01-01 00:00:00 2026-09-29 00:00:00
units
µg/m³    97220
Name: count, dtype: int64
parameter
pm25    50439
pm10    46781
Name: count, dtype: int64
0 duplicate sensor-days
32 negative values
date
2018    35
2019    31
2020    30
2021    30
2022    30
2023     1
2024     1
2025    35
2026    35
Name: location_id, dtype: int64
                                                                              min        max  count
location_id location_name                                    parameter                             
17          R K Puram, Delhi - DPCC                          pm10      2018-01-01 2026-09-29    625
                                                             pm25      2018-01-01 2026-09-29    625
50          Punjabi Bagh, Delhi - DPCC                       pm10      2018-01-01 2026-09-29    619
                                                             pm25      2018-01-01 2026-09-29    620
235         Anand Vihar, New Delhi - DPCC                 

In [ ]:
import requests, numpy as np, pandas as pd

URL = "https://archive-api.open-meteo.com/v1/archive"
HOURLY = ["temperature_2m", "relative_humidity_2m", "dew_point_2m", "wind_speed_10m",
          "wind_gusts_10m", "wind_direction_10m", "surface_pressure", "precipitation", "cloud_cover"]
frames = []
for year in range(2018, 2027):
    end = f"{year}-12-31" if year < 2026 else "2026-09-30"
    r = requests.get(URL, timeout=120, params={
        "latitude": 28.6139, "longitude": 77.2090, "start_date": f"{year}-01-01", "end_date": end,
        "hourly": ",".join(HOURLY), "timezone": "Asia/Kolkata", "wind_speed_unit": "ms"})
    r.raise_for_status()
    frames.append(pd.DataFrame(r.json()["hourly"]))
h = pd.concat(frames, ignore_index=True)
h["time"] = pd.to_datetime(h["time"]); h["date"] = h["time"].dt.normalize()

rad = np.radians(h["wind_direction_10m"])
h["u"] = -h["wind_speed_10m"] * np.sin(rad)      # direction is where wind comes FROM
h["v"] = -h["wind_speed_10m"] * np.cos(rad)

w = h.groupby("date").agg(
    temp_mean=("temperature_2m", "mean"), temp_min=("temperature_2m", "min"), temp_max=("temperature_2m", "max"),
    humidity_mean=("relative_humidity_2m", "mean"), dewpoint_mean=("dew_point_2m", "mean"),
    wind_speed_mean=("wind_speed_10m", "mean"), wind_gust_max=("wind_gusts_10m", "max"),
    pressure_mean=("surface_pressure", "mean"), precip_sum=("precipitation", "sum"),
    cloud_mean=("cloud_cover", "mean"), u=("u", "mean"), v=("v", "mean"),
    hours=("temperature_2m", "count")).reset_index()
w["wind_dir_deg"] = (np.degrees(np.arctan2(-w["u"], -w["v"])) + 360) % 360   # vector mean
w = w.drop(columns=["u", "v"])
w.to_csv("weather_daily.csv", index=False)
print(w["hours"].value_counts().head())   # expect 24; fewer means missing hours, likely the last few days

hours
24    3195
Name: count, dtype: int64


In [ ]:
cal = pd.DataFrame({"date": pd.date_range("2018-01-01", "2026-09-30")})
cal["month"] = cal.date.dt.month
cal["day_of_week"] = cal.date.dt.dayofweek
cal["is_weekend"] = (cal.day_of_week >= 5).astype(int)
cal["season"] = cal.month.map(lambda m: "winter" if m <= 2 else "pre_monsoon" if m <= 5
                              else "monsoon" if m <= 9 else "post_monsoon")   # IMD-style, team decision

In [ ]:
# """Pull daily PM2.5 / PM10 averages for stations around Delhi from the OpenAQ v3 API.

# Setup:   pip install requests
#          export OPENAQ_API_KEY="your-key"      (Windows PowerShell: $env:OPENAQ_API_KEY="your-key")
# Run:     python openaq_pull.py
# Outputs: data/raw/openaq_locations.csv          (station list, for documentation)
#          data/raw/openaq_sensor_days_raw.csv    (one row per sensor per day)

# Free-tier limits are 60 requests/minute and 2,000/hour, so the script sleeps between calls.
# NOT tested against the live API by the author: run with TEST_MODE = True first and
# inspect the output before the full pull.
# """
# import csv
# import os
# import time
# from datetime import datetime, timezone

# import requests

# from google.colab import userdata

# API_KEY = userdata.get("OPENAQ_API_KEY")

# if not API_KEY:
#     raise RuntimeError("OPENAQ_API_KEY not found in Colab Secrets")

# print("API key loaded successfully.")
# BASE = "https://api.openaq.org/v3"
# HEADERS = {"X-API-Key": API_KEY}

# CENTER = "28.6139,77.2090"      # Delhi, "latitude,longitude"
# RADIUS_M = 25000                # API maximum is 25,000 m
# WANTED = {"pm25", "pm10"}       # parameter names as reported by OpenAQ
# TEST_MODE = False           # True = only the first 3 sensors; set False for the full pull

# DATE_FROM = "2018-01-01"
# DATE_TO = "2026-09-30"
# MAX_FIRST = "2019-01-01"   # if this leaves fewer than ~8 locations, loosen to "2021-01-01"
# MIN_LAST = "2026-03-01"    # keep stations still reporting after winter 2025-26


# OUT_DIR = "/content/drive/MyDrive/aqi_project/data/raw"
# LOC_CSV = os.path.join(OUT_DIR, "openaq_locations.csv")
# DAYS_CSV = os.path.join(OUT_DIR, "openaq_sensor_days_raw.csv")


# def get(path, params=None):
#     """GET with rate-limit handling. Returns parsed JSON."""
#     while True:
#         r = requests.get(BASE + path, headers=HEADERS, params=params, timeout=60)
#         if r.status_code == 429:           # too many requests: wait for the next window
#             time.sleep(65)
#             continue
#         r.raise_for_status()
#         remaining = r.headers.get("x-ratelimit-remaining")
#         reset = r.headers.get("x-ratelimit-reset", "60")
#         if remaining is not None and int(remaining) <= 2:
#             time.sleep(int(reset) + 1)
#         else:
#             time.sleep(1.1)
#         return r.json()


# def paged(path, params):
#     """Collect every page of a list endpoint (limit 1000 per page)."""
#     out, page = [], 1
#     while True:

#         j = get(path, {**params, "limit": 1000, "page": page})
#         res = j.get("results", [])
#         out.extend(res)
#         if len(res) < 1000:
#             return out
#         page += 1


# def utc_day(x):
#     """'2026-03-01T...' -> '2026-03-01'; '' if missing."""
#     return ((x or {}).get("utc") or "")[:10]


# def main():
#     os.makedirs(OUT_DIR, exist_ok=True)
#     retrieved_at = datetime.now(timezone.utc).isoformat(timespec="seconds")

#     locations = paged("/locations", {"coordinates": CENTER, "radius": RADIUS_M})
#     print(f"{len(locations)} locations found")

#     loc_rows, active = [], []
#     for loc in locations:
#         coords = loc.get("coordinates") or {}
#         first, last = utc_day(loc.get("datetimeFirst")), utc_day(loc.get("datetimeLast"))
#         loc_rows.append({"location_id": loc.get("id"), "location_name": loc.get("name"),
#                          "latitude": coords.get("latitude"), "longitude": coords.get("longitude"),
#                          "timezone": loc.get("timezone"),
#                          "datetime_first": first, "datetime_last": last})
#         if first and last and first <= MAX_FIRST and last >= MIN_LAST:
#             active.append(loc)

#     with open(LOC_CSV, "w", newline="", encoding="utf-8") as f:
#         w = csv.DictWriter(f, fieldnames=list(loc_rows[0].keys()))
#         w.writeheader()
#         w.writerows(loc_rows)
#     print(f"{len(active)} of {len(locations)} locations are active in the required window")

#     sensors = []
#     for loc in active:
#         for s in loc.get("sensors", []):
#             p = s.get("parameter") or {}
#             if p.get("name") in WANTED:
#                 sensors.append({"location_id": loc.get("id"), "location_name": loc.get("name"),
#                                 "sensor_id": s.get("id"), "parameter": p.get("name"),
#                                 "units": p.get("units")})
#     print(f"{len(sensors)} PM2.5/PM10 sensors in active locations")
#     if TEST_MODE:
#         sensors = sensors[:3]

#     fields = ["location_id", "location_name", "sensor_id", "parameter", "units", "date",
#               "value_avg", "min", "max", "sd", "expected_count", "observed_count",
#               "percent_complete", "retrieved_at"]
#     with open(DAYS_CSV, "w", newline="", encoding="utf-8") as f:
#         w = csv.DictWriter(f, fieldnames=fields)
#         w.writeheader()
#         for i, s in enumerate(sensors, 1):
#             try:
#                 rows = paged(f"/sensors/{s['sensor_id']}/days",
#              {"datetime_from": DATE_FROM + "T00:00:00Z", "datetime_to": DATE_TO + "T23:59:59Z",
#               "date_from": DATE_FROM, "date_to": DATE_TO})
#             except requests.HTTPError as e:
#                 print(f"skip sensor {s['sensor_id']}: {e}")
#                 continue
#             for r in rows:
#               local = ((r.get("period") or {}).get("datetimeFrom") or {}).get("local", "")
#               if not (DATE_FROM <= local[:10] <= DATE_TO):   # guaranteed filter, whatever the API does
#                   continue
#               summ = r.get("summary") or {}
#               cov = r.get("coverage") or {}
#               w.writerow({**s, "date": local[:10], "value_avg": r.get("value"),
#                           "min": summ.get("min"), "max": summ.get("max"), "sd": summ.get("sd"),
#                           "expected_count": cov.get("expectedCount"),
#                           "observed_count": cov.get("observedCount"),
#                           "percent_complete": cov.get("percentComplete"),
#                           "retrieved_at": retrieved_at})
#             print(f"[{i}/{len(sensors)}] sensor {s['sensor_id']} ({s['parameter']}): {len(rows)} days")

# if __name__ == "__main__":
#     main()


API key loaded successfully.
96 locations found
35 of 96 locations are active in the required window
137 PM2.5/PM10 sensors in active locations
[1/137] sensor 399 (pm10): 50 days
[2/137] sensor 12234786 (pm10): 575 days
[3/137] sensor 12234787 (pm25): 575 days
[4/137] sensor 35 (pm25): 50 days
[5/137] sensor 395 (pm10): 49 days
[6/137] sensor 12234795 (pm10): 570 days
[7/137] sensor 12234796 (pm25): 571 days
[8/137] sensor 396 (pm25): 49 days
[9/137] sensor 381 (pm10): 50 days
[10/137] sensor 12235609 (pm10): 576 days
[11/137] sensor 12235610 (pm25): 576 days
[12/137] sensor 384 (pm25): 50 days
[13/137] sensor 14085 (pm10): 47 days
[14/137] sensor 12234701 (pm10): 560 days
[15/137] sensor 14087 (pm25): 47 days
[16/137] sensor 12234702 (pm25): 560 days
[17/137] sensor 12234683 (pm10): 560 days
[18/137] sensor 14620 (pm10): 437 days
[19/137] sensor 12234684 (pm25): 560 days
[20/137] sensor 14912 (pm25): 414 days
[21/137] sensor 15027 (pm10): 1038 days
[22/137] sensor 12234707 (pm10): 540

In [ ]:
"""PM-based Indian National AQI (CPCB breakpoints) for daily 24-hour mean concentrations.

Inputs are 24-hour means in ug/m3. The official CPCB AQI uses up to 8 pollutants
and needs >= 3 of them (one must be PM2.5 or PM10). This project uses PM2.5 and
PM10 only, so values can differ slightly from the CPCB bulletin. Document this.

Breakpoints (conc_lo, conc_hi, index_lo, index_hi) follow the CPCB table.
The top 'Severe' band end-points (PM2.5 380, PM10 510) are the commonly used
convention for reaching 500 - confirm against the CPCB technical document. They do
not affect the AQI > 300 target.
"""

PM25_BANDS = [(0, 30, 0, 50), (31, 60, 51, 100), (61, 90, 101, 200),
              (91, 120, 201, 300), (121, 250, 301, 400), (251, 380, 401, 500)]
PM10_BANDS = [(0, 50, 0, 50), (51, 100, 51, 100), (101, 250, 101, 200),
              (251, 350, 201, 300), (351, 430, 301, 400), (431, 510, 401, 500)]

CATEGORIES = [(50, "Good"), (100, "Satisfactory"), (200, "Moderate"),
              (300, "Poor"), (400, "Very Poor"), (500, "Severe")]


def sub_index(conc, bands):
    if conc is None or conc != conc or conc < 0:
        return None
    conc = int(conc + 0.5)          # round half up: the breakpoint table uses whole numbers
    for c_lo, c_hi, i_lo, i_hi in bands:
        if conc <= c_hi:
            return round(i_lo + (conc - c_lo) * (i_hi - i_lo) / (c_hi - c_lo))
    return 500


def aqi_pm(pm25=None, pm10=None):
    """Overall AQI = highest sub-index. Needs at least one of PM2.5 / PM10."""
    subs = [s for s in (sub_index(pm25, PM25_BANDS), sub_index(pm10, PM10_BANDS))
            if s is not None]
    return max(subs) if subs else None


def category(aqi):
    if aqi is None:
        return None
    for upper, name in CATEGORIES:
        if aqi <= upper:
            return name
    return "Severe"


def is_high_risk(aqi):
    """Project target: AQI above 300 (CPCB 'Very Poor' or 'Severe')."""
    return None if aqi is None else int(aqi > 300)


if __name__ == "__main__":
    # CPCB worked examples: PM2.5 150 -> 323, PM2.5 45 -> 75
    assert sub_index(150, PM25_BANDS) == 323
    assert sub_index(45, PM25_BANDS) == 75
    assert aqi_pm(pm25=120) == 300 and aqi_pm(pm25=121) == 301
    assert aqi_pm(pm10=350) == 300 and aqi_pm(pm10=351) == 301
    assert aqi_pm(pm25=45, pm10=200) == 167  # 101 + 99*99/149 = 166.8
    assert aqi_pm() is None
    print("aqi_utils self-test passed")

aqi_utils self-test passed


In [ ]:

MIN_PCT = 75         # I believe CPCB asks for at least 16 of 24 hours; confirm in its technical document
MIN_STATIONS = 3     # decide after looking at the stations-per-year table

raw = pd.read_csv(DAYS_CSV, parse_dates=["date"])
raw = raw[(raw.units == "µg/m³") & (raw.percent_complete >= MIN_PCT) & (raw.value_avg >= 0)]
# apply your duplicate-station and in_delhi filter to raw here

st = raw.groupby(["location_id", "date", "parameter"])["value_avg"].mean().unstack("parameter").reset_index()
st["aqi_station"] = [aqi_pm(pm25=a, pm10=b) for a, b in zip(st["pm25"], st["pm10"])]

city = st.groupby("date").agg(pm25_mean=("pm25", "mean"), pm10_mean=("pm10", "mean"),
                              n_stations=("aqi_station", "count"), aqi=("aqi_station", "mean")).reset_index()
city = city[city.n_stations >= MIN_STATIONS].copy()
city["aqi"] = city["aqi"].round().astype(int)
city["aqi_category"] = city["aqi"].map(category)
city["high_risk"] = city["aqi"].map(is_high_risk)
city = city.set_index("date").asfreq("D")          # missing days appear as NaN

In [ ]:
raw

,location_id,location_name,sensor_id,parameter,units,date,value_avg,min,max,sd,expected_count,observed_count,percent_complete,retrieved_at
3,17,"R K Puram, Delhi - DPCC",399,pm10,µg/m³,2018-01-04,351.0,165.666667,554.000,121.128044,24,25,104.0,2026-10-02T13:21:07+00:00
4,17,"R K Puram, Delhi - DPCC",399,pm10,µg/m³,2018-01-05,407.0,285.000000,554.000,81.678627,24,24,100.0,2026-10-02T13:21:07+00:00
5,17,"R K Puram, Delhi - DPCC",399,pm10,µg/m³,2018-01-06,407.0,257.000000,622.000,96.767227,24,31,129.0,2026-10-02T13:21:07+00:00
6,17,"R K Puram, Delhi - DPCC",399,pm10,µg/m³,2018-01-07,321.0,200.000000,554.000,104.051811,24,24,100.0,2026-10-02T13:21:07+00:00
7,17,"R K Puram, Delhi - DPCC",399,pm10,µg/m³,2018-01-08,275.0,147.000000,438.000,93.923225,24,19,79.0,2026-10-02T13:21:07+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
97215,10921,"NISE Gwal Pahari, Gurugram - IMD",12235707,pm25,µg/m³,2026-09-25,17.5,3.147500,39.845,12.229165,24,24,100.0,2026-10-02T13:21:07+00:00
97216,10921,"NISE Gwal Pahari, Gurugram - IMD",12235707,pm25,µg/m³,2026-09-26,13.8,4.532500,29.000,6.178250,24,23,96.0,2026-10-02T13:21:07+00:00
97217,10921,"NISE Gwal Pahari, Gurugram - IMD",12235707,pm25,µg/m³,2026-09-27,15.7,2.735000,53.305,14.615478,24,18,75.0,2026-10-02T13:21:07+00:00
97218,10921,"NISE Gwal Pahari, Gurugram - IMD",12235707,pm25,µg/m³,2026-09-28,20.0,2.075000,41.260,11.091789,24,24,100.0,2026-10-02T13:21:07+00:00


In [ ]:
cal = pd.DataFrame({"date": pd.date_range("2018-01-01", "2026-09-30")})
cal["month"] = cal.date.dt.month
cal["day_of_week"] = cal.date.dt.dayofweek
cal["is_weekend"] = (cal.day_of_week >= 5).astype(int)
cal["season"] = cal.month.map(lambda m: "winter" if m <= 2 else "pre_monsoon" if m <= 5
                              else "monsoon" if m <= 9 else "post_monsoon")   # IMD-style, team decision
cal.shape

(3195, 5)

In [ ]:
import pandas as pd
import numpy as np

B = "/content/drive/MyDrive/aqi_project/data/raw/"

raw = pd.read_csv(
    B + "openaq_sensor_days_raw.csv",
    parse_dates=["date"]
)

loc = pd.read_csv(
    B + "openaq_locations.csv"
)

weather = pd.read_csv(
    "/content/weather_daily.csv",
    parse_dates=["date"]
)

cal = pd.read_csv(
    "/content/cal.csv",
    parse_dates=["date"]
)

# Diwali dates: 2018–2025
diwali = pd.to_datetime([
    "2018-11-07",
    "2019-10-27",
    "2020-11-14",
    "2021-11-04",
    "2022-10-24",
    "2023-11-12",
    "2024-10-31",
    "2025-10-20"
])

# Festival window: ±5 days around Diwali
N = 5

diff = (
    cal["date"].values[:, None] - diwali.values[None, :]
) / np.timedelta64(1, "D")

nearest_idx = np.abs(diff).argmin(axis=1)

cal["days_from_diwali"] = diff[
    np.arange(len(diff)),
    nearest_idx
]

cal["is_festival_window"] = (
    cal["days_from_diwali"].abs() <= N
).astype(int)

In [ ]:
cal_path = "/content/cal.csv"

cal.to_csv(cal_path, index=False)

print(f"Saved: {cal_path}")
print(f"Shape: {cal.shape}")

Saved: /content/cal.csv
Shape: (3195, 7)
